# Hyperparameter tuning

### Importing the required libraries

In [ ]:
!pip install tensorflow
!pip install -q -U keras-tuner

In [ ]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

### Define the method that returns the model

In [ ]:
def model_builder(hp):
    model=keras.Sequential()
    model.add(keras.layers.Flatten())

    hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
    model.add(keras.layers.Dense(units=hp_units, activation='relu'))
    model.add(keras.layers.Dense(10, activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy'])
    
    return model

### Instantiate the tuner

In [ ]:
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy', #aim for the best validation accuracy
    max_epochs=10,
    factor=3, #take the top 33% of the models in each round
    directory='my_dir',
    project_name='intro_to_kt'
)

stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
#stop training if the validation loss does not improve for 5 epochs

### Find the best hyperparameters

In [ ]:
tuner.search(X_train, y_train, epochs=50, validation_split=0.2, callbacks=[stop_early])
#train different models for a total of 50 epochs (overrides max_epochs)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
#retirn the top1 hyperparameter set

print(f"""
The hyperparameter search is complete. The optimal number of units in the first densely-connected
layer is {best_hps.get('units')} and the optimal learning rate for the optimizer
is {best_hps.get('learning_rate')}.
""")

### Find the best epoch to train the model with the best hyperparameters

In [ ]:
model = tuner.hypermodel.build(best_hps)
#build the model defined in model_builder with the best hyperparameters

history = model.fit(X_train, y_train, epochs=50, validation_split=0.2)

val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % (best_epoch,))

### Train the model with the best number of epochs

In [ ]:
#train the model with the best number of epochs
hypermodel = tuner.hypermodel.build(best_hps)
hypermodel.fit(X_train, y_train, epochs=best_epoch, validation_split=0.2)
eval_result = hypermodel.evaluate(X_test, y_test)
print("[test loss, test accuracy]:", eval_result)